In [ ]:
# This project consists of 2 parts:
# main part - code for finding the best model for the current dataset. Takes quite a long time to run
# final part - code that uses the best model and saves the result

In [ ]:
# main part
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from catboost import CatBoostClassifier

# Importing libraries and tools used for data analysis, feature engineering, 
# model training, and evaluation.

'''
Third Data Science study project.
The project includes data analysis, missing value imputation, 
feature engineering, training of several classification models, 
and comparison of their results.
'''

# Initial data analysis and exploration
test = pd.read_csv("test.csv")
train = pd.read_csv("train.csv")
train_raw = pd.read_csv("train.csv")

train.shape
test.shape
train.head()
test.head()
train.info()
test.info()

train[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].describe()
train.head()
train.isna().sum()

train["Transported"].value_counts(normalize=True).plot(kind="bar")
plt.title("Transported class distribution")
plt.xlabel("Transported")
plt.ylabel("Proportion")
plt.show()

plt.hist(train["Age"], bins=30)
plt.title("Age distribution")
plt.xlabel("Age")
plt.ylabel("Number of passengers")
plt.show()

plt.hist(train["RoomService"], bins=30)
plt.title("RoomService distribution")
plt.xlabel("RoomService")
plt.ylabel("Number of passengers")
plt.show()

transported_rate = train.groupby("CryoSleep")["Transported"].mean()
transported_rate.plot(kind="bar")
plt.title("Transported rate by CryoSleep")
plt.xlabel("CryoSleep")
plt.ylabel("Transported rate")
plt.ylim(0, 1)
plt.show()

transported_rate = train.groupby("HomePlanet")["Transported"].mean()
transported_rate.plot(kind="bar")
plt.title("Transported rate by HomePlanet")
plt.xlabel("HomePlanet")
plt.ylabel("Transported rate")
plt.ylim(0, 1)
plt.show()

'''
Based on the plots, the following observations can be made:
- The majority of passengers fall within the average age range.
- The Age distribution does not look strictly normal.
- There are isolated observations in the extreme age ranges.

These plots are used for an initial understanding of the data structure, 
not for strict statistical distribution testing.
'''

spend_cols = [
    # Features reflecting passenger spending on ship services
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

train[train[spend_cols].isna().any(axis=1)][['CryoSleep'] + spend_cols].head(20)

'''
The data shows a strong relationship between CryoSleep and spending on ship services:
passengers with CryoSleep=True mostly have no expenses, 
while passengers with CryoSleep=False have expenses much more frequently.

Therefore, it is logical to use different approaches for missing spending values:
for CryoSleep=True, fill missing values with 0;
for CryoSleep=False, use the median values of the corresponding feature.
'''

# Cleaning data
train.groupby('CryoSleep')[spend_cols].apply(lambda x: x.isna().mean())
train[train["CryoSleep"] == False][spend_cols].isna().sum()
train.loc[train["CryoSleep"] == False, "RoomService"].median()
train.groupby("CryoSleep")[spend_cols].median()

mask_true = train["CryoSleep"] == True
mask_false = train["CryoSleep"] == False
train.loc[mask_true, spend_cols] = (train.loc[mask_true, spend_cols].fillna(0))

for col in spend_cols:
    median = train.loc[mask_false, col].median()
    train.loc[mask_false, col] = train.loc[mask_false, col].fillna(median)

train[spend_cols].isna().sum()
train.loc[mask_true, spend_cols].describe()
train.loc[mask_false, spend_cols].describe()
train[train["CryoSleep"].isna()][["CryoSleep"] + spend_cols]

'''
For rows with missing CryoSleep, we use spending information:
if there are expenses, we assume CryoSleep=False;
if all expenses are missing or known values equal zero, we consider CryoSleep=True.
Remaining ambiguous cases are filled separately.
'''

cryo_na = train["CryoSleep"].isna()
total_spend = train[spend_cols].fillna(0).sum(axis=1)
total_spend[cryo_na].gt(0).value_counts()

total_spend = train[spend_cols].fillna(0).sum(axis=1)
train.loc[cryo_na & (total_spend == 0), spend_cols].isna().sum()
train.loc[cryo_na & (total_spend == 0), spend_cols].head(20)

total_spend = train[spend_cols].fillna(0).sum(axis=1)
mask_zero_spend = cryo_na & (total_spend == 0)
train.loc[mask_zero_spend, spend_cols].isna().any(axis=1).value_counts()

train.loc[
    cryo_na & (total_spend == 0) & train[spend_cols].isna().any(axis=1),
    ["CryoSleep"] + spend_cols + ["Age", "VIP", "HomePlanet", "Destination", "Cabin"]
]

total_spend = train[spend_cols].fillna(0).sum(axis=1)
mask_false = cryo_na & (total_spend > 0)
train.loc[mask_false, "CryoSleep"] = False
train["CryoSleep"].value_counts(dropna=False)

mask_true = (
    train["CryoSleep"].isna()
    & train[spend_cols].notna().all(axis=1)
    & (train[spend_cols].sum(axis=1) == 0)
)
train.loc[mask_true, "CryoSleep"] = True
train["CryoSleep"].value_counts(dropna=False)

mask_11 = (
    train["CryoSleep"].isna()
    & (train[spend_cols].fillna(0).sum(axis=1) == 0)
)
train.loc[mask_11, spend_cols] = (train.loc[mask_11, spend_cols].fillna(0))
train[spend_cols].isna().sum()
train["CryoSleep"].isna().sum()
train.loc[train["CryoSleep"] == False, spend_cols].isna().sum()

mask_false = train["CryoSleep"] == False
for col in spend_cols:
    median = train.loc[mask_false, col].median()
    train.loc[mask_false, col] = train.loc[mask_false, col].fillna(median)

train.loc[train["CryoSleep"] == False, spend_cols].isna().sum()

train["Age"].describe()
train["Age"].isna().sum()
train["Age"] = train["Age"].fillna(train["Age"].median())
train["Age"].isna().sum()

train["VIP"].value_counts(dropna=False)
train["VIP"].isna().sum()
train.groupby("VIP")[spend_cols].median()
train.groupby("VIP")["Transported"].mean()

'''
There is a correlation between VIP and spending, but it is not absolute.
Therefore, the presence of spending can be used as an additional feature 
when analyzing missing VIP values, but it cannot be considered an absolute rule 
for imputing VIP. For the current experiment, missing VIP values are filled with False.
'''

mask_vip_na = train_raw["VIP"].isna()
has_spend = (train_raw.loc[mask_vip_na, spend_cols].gt(0).any(axis=1))
has_spend.value_counts()

has_spend_all = (train_raw[spend_cols].gt(0).any(axis=1))
pd.crosstab(train_raw["VIP"], has_spend_all, normalize="index")
train_raw["VIP"].value_counts(normalize=True, dropna=False)

train["VIP"] = train["VIP"].fillna(False)
train["HasSpend"] = (train[spend_cols].sum(axis=1) > 0).astype(int)
train_raw["VIP"].value_counts(normalize=True, dropna=False)
train["VIP"].isna().sum()
train.isna().sum()

train["HomePlanet"].value_counts(dropna=False)
train["Destination"].value_counts(dropna=False)
pd.crosstab(train["HomePlanet"], train["Destination"], normalize="index")

cabin_deck = train["Cabin"].str.split("/").str[0]
pd.crosstab(train["HomePlanet"], cabin_deck, normalize="index")

train["Deck"] = train["Cabin"].str.split("/").str[0].astype("string")
pd.crosstab(train["Deck"], train["HomePlanet"], normalize="index")

transported_rate = train.groupby("Deck")["Transported"].mean()
transported_rate.plot(kind="bar")
plt.title("Transported rate by Deck")
plt.xlabel("Deck")
plt.ylabel("Transported rate")
plt.ylim(0, 1)
plt.show()

train.loc[train["HomePlanet"].isna(), ["HomePlanet", "Deck"]]["Deck"].value_counts(dropna=False)

deck_to_planet = {
    "A": "Europa",
    "B": "Europa",
    "C": "Europa",
    "G": "Earth",
    "T": "Europa"
}

pd.crosstab(train.loc[train["Deck"] == "F", "Destination"], train.loc[train["Deck"] == "F", "HomePlanet"], normalize="index")
pd.crosstab(train.loc[train["Deck"] == "E", "Destination"], train.loc[train["Deck"] == "E", "HomePlanet"], normalize="index")
pd.crosstab(train.loc[train["Deck"] == "D", "Destination"], train.loc[train["Deck"] == "D", "HomePlanet"], normalize="index")

mask_planet = train["HomePlanet"].isna()
deck = train["Cabin"].str.split("/").str[0]
planet = deck.map(deck_to_planet).fillna("Unknown")
train.loc[mask_planet, "HomePlanet"] = planet[mask_planet]

train["HomePlanet"].isna().sum()
train["HomePlanet"].value_counts(dropna=False)
pd.crosstab(train["HomePlanet"], train["Destination"], normalize="index")

train["Destination"].isna().sum()
train.loc[train["Destination"].isna(), "HomePlanet"].value_counts(dropna=False)
train["Destination"] = train["Destination"].fillna("Unknown")
train.isna().sum()

train["Cabin"].head()
train['CabinNumber'] = train["Cabin"].str.split("/").str[1]
train['Side'] = train["Cabin"].str.split("/").str[2].astype("string")
train['CabinNumber'].head()
train['Side'].head()

train["CabinNumber"] = pd.to_numeric(train["CabinNumber"])
train[["CabinNumber", "Side", "Deck"]].dtypes
train["CabinNumber"].isna().sum()
train["Side"].value_counts(dropna=False)
train["CabinNumber"].describe()
train["CabinNumber"].nunique()
train["CabinNumber"].quantile([0, .1, .25, .5, .75, .9, .99, 1])

train["CabinNumberMissing"] = train["CabinNumber"].isna().astype(int)
train["CabinNumber"] = train["CabinNumber"].fillna(train["CabinNumber"].median())

train["Deck"] = train["Deck"].fillna("Unknown")
train["Side"] = train["Side"].fillna("Unknown")

transported_rate = train.groupby("Deck")["Transported"].mean()
transported_rate.plot(kind="bar")
plt.title("Transported rate by Deck")
plt.xlabel("Deck")
plt.ylabel("Transported rate")
plt.ylim(0, 1)
plt.show()

train.isna().sum()

cols_to_drop = ["Cabin", "Name", "PassengerId"]
train = train.drop(columns=cols_to_drop, errors="ignore")
train["CryoSleep"] = train["CryoSleep"].fillna(train["CryoSleep"].mode()[0])
train.isna().sum()
train.dtypes

numeric_cols = [
    'Age', 'HasSpend', 'CabinNumber', 'CabinNumberMissing',
    'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'
]

categorical_cols = [
    'HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side'
]

target = 'Transported'

x = train.drop(columns="Transported")
y = train["Transported"]

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)
x_train.shape
x_val.shape
y_train.value_counts(normalize=True)
y_val.value_counts(normalize=True)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

liner_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

liner_model.fit(x_train, y_train)

y_pred = liner_model.predict(x_val)
accuracy = accuracy_score(y_val, y_pred)
print(accuracy)

print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

results = x_val.copy()
results["actual"] = y_val
results["predicted"] = y_pred
results["error"] = results["actual"] != results["predicted"]

results.groupby("CryoSleep")["error"].mean()
results.groupby("HomePlanet")["error"].mean()
results.groupby("Deck")["error"].mean()
results.groupby("VIP")["error"].mean()
results.groupby("Side")["error"].mean()

results["AgeGroup"] = pd.cut(results["Age"], bins=[0, 12, 18, 30, 50, 80])
results.groupby("AgeGroup", observed=True)["error"].mean()

# ERROR PLOT BY AGE GROUPS
age_error = results.groupby("AgeGroup", observed=True)["error"].mean()
plt.figure(figsize=(8, 5))
age_error.plot(kind="bar")
plt.title("Logistic Regression Error Rate by Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Error rate")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

'''
Error Analysis Logistic Regression:
- The average error rate on the validation set is about 21%.
- Differences in error rate for CryoSleep and Side are relatively small.
- A higher error rate is observed for some HomePlanet and Deck categories.
- Increased error rates are also observed for certain age groups.
- When interpreting differences, the size of each group must be taken into account.

Conclusion:
Errors are unevenly distributed among groups.
The most noticeable differences are observed in HomePlanet, Deck, and age groups.
These groups can be further investigated in subsequent steps.
'''

results.groupby("AgeGroup", observed=True).agg(error_rate=("error", "mean"), count=("error", "size"))
results.groupby("Deck").agg(error_rate=("error", "mean"), count=("error", "size")).sort_values("error_rate", ascending=False)

train["Child"] = (train["Age"] <= 12).astype(int)
numeric_cols.append("Child")

x = train.drop(columns="Transported")
y = train["Transported"]
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

print("Child in x_train:", "Child" in x_train.columns)
print("Child in x_val:", "Child" in x_val.columns)
print("Child in numeric_cols:", "Child" in numeric_cols)
print(x_train["Child"].value_counts())

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

liner_model_e1 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

liner_model_e1.fit(x_train, y_train)
y_pred = liner_model_e1.predict(x_val)
accuracy = accuracy_score(y_val, y_pred)
print(accuracy)

print("Child in x_train:", "Child" in x_train.columns)
print("Child in x_val:", "Child" in x_val.columns)
print("Child in numeric_cols:", "Child" in numeric_cols)

# Adding the Child feature barely changed the accuracy on the validation set.
# In the current experiments, we keep the original feature set.
numeric_cols.remove("Child")
train = train.drop(columns=["Child"])
print("Child" in train.columns)
print("Child" in numeric_cols)

x = train.drop(columns="Transported")
y = train["Transported"]
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

rf_model_e2 = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(random_state=42, n_estimators=100))
])

rf_model_e2.fit(x_train, y_train)
y_pred = rf_model_e2.predict(x_val)
accuracy_rf = accuracy_score(y_val, y_pred)
print(accuracy_rf)

# On this validation set, Random Forest showed higher accuracy than Logistic Regression.

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

param_dist = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [None, 5, 10, 15, 20, 30],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4, 8],
    "classifier__max_features": ["sqrt", "log2", None]
}

rf_model_e3 = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(random_state=42))
])

random_search_rf = RandomizedSearchCV(
    estimator=rf_model_e3,
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search_rf.fit(x_train, y_train)
best_rf = random_search_rf.best_estimator_
y_pred_tuned = best_rf.predict(x_val)
accuracy_tuned = accuracy_score(y_val, y_pred_tuned)

print(accuracy_tuned)
print(random_search_rf.best_params_)
print(random_search_rf.best_score_)

rf_cv_scores = cross_val_score(rf_model_e2, x_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
print(rf_cv_scores)
print(rf_cv_scores.mean())
print(rf_cv_scores.std())

tuned_cv_scores = cross_val_score(random_search_rf.best_estimator_, x_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
print(tuned_cv_scores)
print(tuned_cv_scores.mean())
print(tuned_cv_scores.std())

'''
Tuned Random Forest:
- Baseline RF:
  mean CV = 0.7954
  std = 0.0027
- Tuned RF:
  mean CV = 0.8035
  std = 0.0067

After tuning, the mean quality on cross-validation increased.
At the same time, the standard deviation between splits increased, 
meaning the results became less stable.
On the validation set, accuracy also increased from 0.7956 to 0.8033.

Conclusion:
Tuning provided a quality boost on the considered splits, 
but it was accompanied by an increase in result variance.
'''

y_pred_rf = rf_model_e2.predict(x_val)
print(confusion_matrix(y_val, y_pred_rf))
print(classification_report(y_val, y_pred_rf))

rf_results = x_val.copy()
rf_results["actual"] = y_val
rf_results["predicted"] = y_pred_rf
rf_results["error"] = (rf_results["actual"] != rf_results["predicted"])
rf_results["error"].mean()

rf_results.groupby("CryoSleep")["error"].mean()
rf_results.groupby("HomePlanet")["error"].mean()
rf_results["HomePlanet"].value_counts()
rf_results.groupby("Deck")["error"].mean()
rf_results["Deck"].value_counts()
rf_results.groupby("VIP")["error"].mean()
rf_results.groupby("Side")["error"].mean()

sort_age = [0, 15, 25, 50, 75]
sort_age_name = ['up to 15', 'up to 25', 'up to 50', 'up to 75']
rf_results['Age_group'] = pd.cut(
    rf_results['Age'],
    bins=sort_age,
    labels=sort_age_name,
    right=True,        
    include_lowest=True
)
rf_results.groupby("Age_group")["error"].mean()

# ERROR ANALYSIS PLOTS FOR RANDOM FOREST
error_features_rf = ["CryoSleep", "HomePlanet", "Deck", "VIP", "Side"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, feature in zip(axes, error_features_rf):
    error_rate = rf_results.groupby(feature)["error"].mean().sort_values(ascending=False)
    error_rate.plot(kind="bar", ax=ax)
    ax.set_title(f"Error rate by {feature}")
    ax.set_xlabel(feature)
    ax.set_ylabel("Error rate")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)

axes[-1].axis("off")
plt.suptitle("Error Analysis — Random Forest", fontsize=16)
plt.tight_layout()
plt.show()

age_error_rf = rf_results.groupby("Age_group", observed=True)["error"].mean()
plt.figure(figsize=(8, 5))
age_error_rf.plot(kind="bar")
plt.title("Random Forest Error Rate by Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Error rate")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

'''
Brief summary of Random Forest Error Analysis:
1) The average error rate is about 19.7%.
2) CryoSleep: error rate for False is higher than for True. The model makes more mistakes among passengers without CryoSleep.
3) HomePlanet: the highest error rate is observed for Earth and Unknown. For Unknown, the small group size must be considered, so this value cannot be directly compared to larger categories.
4) Deck: an increased error rate is observed for specific categories, particularly G and E. Group sizes are considered during interpretation.
5) VIP: error rate for VIP=False is higher than for VIP=True. However, categories have different sizes, so comparison should be viewed alongside observation counts.
6) Age_group: the highest error rate is observed in the youngest age group. This may indicate that the model struggles to separate classes among younger passengers.
7) Confusion matrix:
   [[1115  180]
    [ 333  980]]
   For the True class, the number of FN (333) is higher than FP (180). That is, among passengers who were actually Transported, the model more often predicted False than it predicted True among those actually not Transported.
8) Classification report shows:
   False: precision = 0.77, recall = 0.86
   True:  precision = 0.84, recall = 0.75
   For the False class, recall is higher than for True. For the True class, precision is higher than recall.

Generalization:
The differences in errors by HomePlanet, Deck, and age groups are of the greatest interest for further analysis. At the same time, results for small groups must be interpreted with caution.
'''

perm_importance = permutation_importance(
    rf_model_e2, x_val, y_val, scoring="accuracy", n_repeats=10, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": x_val.columns,
    "importance_mean": perm_importance.importances_mean,
    "importance_std": perm_importance.importances_std
})
importance_df = importance_df.sort_values("importance_mean", ascending=False)
importance_df

# PERMUTATION IMPORTANCE PLOT — RANDOM FOREST
top_n = 15
plot_data_rf = importance_df.head(top_n).sort_values("importance_mean")

plt.figure(figsize=(10, 7))
plt.barh(plot_data_rf["feature"], plot_data_rf["importance_mean"])
plt.xlabel("Mean decrease in accuracy")
plt.ylabel("Feature")
plt.title("Permutation Importance — Random Forest")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

'''
Permutation Importance shows how much changing a feature's values affects the model's accuracy.
In the current Random Forest, individual features, such as Age and Deck, make the most noticeable contribution.
However, high feature importance does not mean the model works equally well on all its categories. 
For example, the model may actively use Deck for predictions while simultaneously having an increased error rate on specific Decks.
Features with very low permutation importance can be tested in subsequent experiments by removing them and re-running cross-validation.
'''

X_train_e5 = x_train.drop(columns=["CabinNumberMissing"])
X_val_e5 = x_val.drop(columns=["CabinNumberMissing"])

numeric_cols_e5 = numeric_cols.copy()
numeric_cols_e5.remove("CabinNumberMissing")

preprocessor_rf_e5 = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols_e5),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

rf_model_e5 = Pipeline([
    ("preprocessor", preprocessor_rf_e5),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

scores_e5 = cross_val_score(rf_model_e5, X_train_e5, y_train, cv=5, scoring="accuracy", n_jobs=-1)
print(scores_e5)
print(scores_e5.mean())
print(scores_e5.std())

rf_model_e5.fit(X_train_e5, y_train)
y_pred_e5 = rf_model_e5.predict(X_val_e5)
accuracy_e5 = accuracy_score(y_val, y_pred_e5)
print(accuracy_e5)

'''
Removing CabinNumberMissing did not provide a consistent quality improvement.
The mean CV value changed slightly after removing the feature, 
but there is no reason to consider this feature removal as a significant model improvement.
We keep the original feature set in the current experiment.
'''

x_train_e6 = x_train.drop(columns=["VIP"])
x_val_e6 = x_val.drop(columns=["VIP"])

numeric_cols_e6 = numeric_cols.copy()
categorical_cols_e6 = categorical_cols.copy()
categorical_cols_e6.remove("VIP")

preprocessor_rf_e6 = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols_e6),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols_e6)
    ]
)

rf_model_e6 = Pipeline([
    ("preprocessor", preprocessor_rf_e6),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

scores_e6 = cross_val_score(rf_model_e6, x_train_e6, y_train, cv=5, scoring="accuracy", n_jobs=-1)
print(scores_e6)
print(scores_e6.mean())
print(scores_e6.std())

cv_repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

scores_e4_repeated = cross_val_score(rf_model_e2, x_train, y_train, cv=cv_repeated, scoring="accuracy", n_jobs=-1)
print(scores_e4_repeated)
print("mean:", scores_e4_repeated.mean())
print("std:", scores_e4_repeated.std())

scores_e6_repeated = cross_val_score(rf_model_e6, x_train_e6, y_train, cv=cv_repeated, scoring="accuracy", n_jobs=-1)
print(scores_e6_repeated)
print("mean:", scores_e6_repeated.mean())
print("std:", scores_e6_repeated.std())

# Removing VIP practically did not change the mean quality on Repeated Stratified K-Fold.
# At the same time, the standard deviation increased slightly, 
# so no noticeable advantage is observed from removing the feature.

# E7 — compare baseline RF and tuned RF on the same 15 splits of Repeated Stratified K-Fold
cv_repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

scores_tuned_repeated = cross_val_score(random_search_rf.best_estimator_, x_train, y_train, cv=cv_repeated, scoring="accuracy", n_jobs=-1)
print(scores_tuned_repeated)
print("mean:", scores_tuned_repeated.mean())
print("std:", scores_tuned_repeated.std())

'''
On Repeated Stratified K-Fold, tuned Random Forest shows a higher mean accuracy compared to baseline RF.
However, the standard deviation also increased, indicating greater variance between splits.
Thus, tuning gave a slight mean quality boost but did not improve model stability.
'''

# Moving on to CatBoost

# Checking feature types
x_train[categorical_cols].dtypes
x_train[numeric_cols].dtypes

# Preparing data for CatBoost
X_train_cat = x_train.copy()
X_val_cat = x_val.copy()

for col in categorical_cols:
    X_train_cat[col] = X_train_cat[col].astype("string")
    X_val_cat[col] = X_val_cat[col].astype("string")

X_train_cat[categorical_cols].dtypes

# CatBoost baseline
cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="Logloss",
    cat_features=categorical_cols,
    random_seed=42,
    verbose=False
)

cat_model.fit(X_train_cat, y_train)
y_pred_cat = cat_model.predict(X_val_cat)
accuracy_cat = accuracy_score(y_val, y_pred_cat)
print("CatBoost accuracy:", accuracy_cat)

# Preparing data for Repeated Stratified K-Fold
X_train_cat_cv = x_train.copy()
for col in categorical_cols:
    X_train_cat_cv[col] = X_train_cat_cv[col].astype("string")

# CatBoost Repeated Stratified K-Fold
scores_cat_repeated = []

for train_idx, val_idx in cv_repeated.split(X_train_cat_cv, y_train):
    X_fold_train = X_train_cat_cv.iloc[train_idx]
    X_fold_val = X_train_cat_cv.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    cat_model_cv = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        cat_features=categorical_cols,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=1
    )

    cat_model_cv.fit(X_fold_train, y_fold_train)
    pred = cat_model_cv.predict(X_fold_val)
    score = accuracy_score(y_fold_val, pred)
    scores_cat_repeated.append(score)

print("CatBoost CV scores:")
print(scores_cat_repeated)
print("mean:", np.mean(scores_cat_repeated))
print("std:", np.std(scores_cat_repeated))

res_cat = X_val_cat.copy()
y_pred_cat = cat_model.predict(X_val_cat)
res_cat["actual"] = y_val
res_cat["predicted"] = y_pred_cat

# Error by CatBoost
res_cat["error"] = (res_cat["actual"] != res_cat["predicted"])
print("Mean error rate:", res_cat["error"].mean())
print("\nNumber of correct/incorrect predictions:")
print(res_cat["error"].value_counts())

# Errors by CryoSleep
print("\nError rate by CryoSleep:")
print(res_cat.groupby("CryoSleep")["error"].mean())

# Errors by HomePlanet
print("\nError rate by HomePlanet:")
print(res_cat.groupby("HomePlanet")["error"].mean())

# Errors by Deck
print("\nError rate by Deck:")
print(res_cat.groupby("Deck")["error"].mean())

# Errors by VIP
print("\nError rate by VIP:")
print(res_cat.groupby("VIP")["error"].mean())

# Errors by Side
print("\nError rate by Side:")
print(res_cat.groupby("Side")["error"].mean())

# Age groups
sort_age = [0, 15, 25, 50, 75]
sort_age_name = ["up to 15", "up to 25", "up to 50", "up to 75"]

res_cat["Age_group"] = pd.cut(
    res_cat["Age"],
    bins=sort_age,
    labels=sort_age_name,
    right=True,
    include_lowest=True
)

print("\nError rate by age groups:")
print(res_cat.groupby("Age_group", observed=True)["error"].mean())
res_cat.groupby("Age_group")["error"].mean()

y_pred_cat = cat_model.predict(X_val_cat)
print(confusion_matrix(y_val, y_pred_cat))
print(classification_report(y_val, y_pred_cat))

'''
Brief summary of Error Analysis for CatBoost:
1) The average error rate is about 21.1%.
2) Number of correct and incorrect predictions: correct = 2057, errors = 551.
3) CryoSleep: error rate for False is higher than for True, similar to Random Forest results.
4) HomePlanet: an increased error rate remains for Earth. For Unknown, the small group size must be considered.
5) Deck: the highest error rate is observed for specific categories, especially G and E. This pattern was also observed in Random Forest.
6) VIP and Side: the error distribution remains close to the results of the previous model.
7) Age_group: an increased error rate remains among the youngest age groups.
8) Confusion matrix:
   [[1051  244]
    [ 255 1058]]
   The number of FP and FN became more balanced compared to Random Forest. For the True class, recall is 0.81.
9) Classification report shows: precision and recall for both classes are at approximately the same level — 0.81.

Generalization:
Certain passenger groups, in particular specific Deck categories, HomePlanet=Earth, and younger age groups, remain relatively difficult to classify across different models.
This may indicate peculiarities in the data distribution within these groups, rather than just the specifics of a particular algorithm.
'''

perm_cat = permutation_importance(
    cat_model, X_val_cat, y_val, scoring="accuracy", n_repeats=10, random_state=42, n_jobs=-1
)

importance_cat = pd.DataFrame({
    "feature": X_val_cat.columns,
    "importance_mean": perm_cat.importances_mean,
    "importance_std": perm_cat.importances_std
})
importance_cat = importance_cat.sort_values("importance_mean", ascending=False)
importance_cat

'''
Compared to Random Forest, the permutation importance distribution in CatBoost is different.
In CatBoost, the contribution of some spending-related features, such as Spa and RoomService, is more noticeable, 
while the relative importance of CryoSleep, Deck, and HomePlanet is lower.
This shows that models use features somewhat differently. 
However, permutation importance does not show causality and reflects feature contribution strictly within the specific trained model.
'''

# Moving on to CatBoost tuning
param_dist_cat = {
    "depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "iterations": [200, 300, 500, 700]
}

cat_model_tun = CatBoostClassifier(
    loss_function="Logloss",
    cat_features=categorical_cols,
    random_seed=42,
    verbose=False
)

search_result = cat_model_tun.randomized_search(
    param_dist_cat,
    X=X_train_cat,
    y=y_train,
    cv=5,
    n_iter=20,
    partition_random_seed=42,
    stratified=True,
    search_by_train_test_split=False,
    verbose=False
)

best_params_cat = search_result["params"]
print("Best CatBoost parameters:")
print(best_params_cat)

cat_model_tuned = CatBoostClassifier(
    **best_params_cat,
    loss_function="Logloss",
    cat_features=categorical_cols,
    random_seed=42,
    verbose=False
)

cat_model_tuned.fit(X_train_cat, y_train)
y_pred_cat_tuned = cat_model_tuned.predict(X_val_cat)
accuracy_cat_tuned = accuracy_score(y_val, y_pred_cat_tuned)
print(accuracy_cat_tuned)

scores_cat_tuned = []

for train_idx, val_idx in cv_repeated.split(X_train_cat_cv, y_train):
    X_fold_train = X_train_cat_cv.iloc[train_idx]
    X_fold_val = X_train_cat_cv.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    model = CatBoostClassifier(
        **best_params_cat,
        loss_function="Logloss",
        cat_features=categorical_cols,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=1
    )

    model.fit(X_fold_train, y_fold_train)
    pred = model.predict(X_fold_val)
    score = accuracy_score(y_fold_val, pred)
    scores_cat_tuned.append(score)

print("Tuned CatBoost CV scores:")
print(scores_cat_tuned)
print("mean:", np.mean(scores_cat_tuned))
print("std:", np.std(scores_cat_tuned))

res_cat_tuned = X_val_cat.copy()

y_pred_cat_tuned = cat_model_tuned.predict(X_val_cat)

res_cat_tuned["actual"] = y_val
res_cat_tuned["predicted"] = y_pred_cat_tuned

res_cat_tuned["error"] = (
    res_cat_tuned["actual"] != res_cat_tuned["predicted"]
)

print("Mean error rate:")
print(res_cat_tuned["error"].mean())

print("\nNumber of correct/incorrect predictions:")
print(res_cat_tuned["error"].value_counts())

print("\nError rate by CryoSleep:")
print(res_cat_tuned.groupby("CryoSleep")["error"].mean())

print("\nError rate by HomePlanet:")
print(res_cat_tuned.groupby("HomePlanet")["error"].mean())

print("\nError rate by Deck:")
print(res_cat_tuned.groupby("Deck")["error"].mean())

print("\nError rate by VIP:")
print(res_cat_tuned.groupby("VIP")["error"].mean())

print("\nError rate by Side:")
print(res_cat_tuned.groupby("Side")["error"].mean())


# Creating age groups
sort_age = [0, 15, 25, 50, 75]

sort_age_name = [
    "up to 15",
    "up to 25",
    "up to 50",
    "up to 75"
]

res_cat_tuned["Age_group"] = pd.cut(
    res_cat_tuned["Age"],
    bins=sort_age,
    labels=sort_age_name,
    right=True,
    include_lowest=True
)

print("\nError rate by age groups:")
print(
    res_cat_tuned
    .groupby("Age_group", observed=True)["error"]
    .mean()
)


'''
Error Analysis for Tuned CatBoost:

- The average error rate on the validation set is about 18.7%.
- Correct predictions: 2121
- Incorrect predictions: 487

CryoSleep:
Differences in error rate between categories are small:
False — about 18.9%, True — about 18.2%

HomePlanet:
The highest error rate is observed for Earth (about 26.3%).
For Europa and Mars, the error rate is noticeably lower.
The result for Unknown should be interpreted with caution
due to the small number of observations.

Deck:
The highest error rate is observed for G (about 28.7%),
followed by E (about 22.7%).
For B and C, the error rate is significantly lower.
Category T must be considered with caution due to the small
number of observations.

VIP:
Error rate for VIP=False is about 18.9%,
for VIP=True — about 7.0%.
When comparing, the different group sizes must be taken into account.

Side:
Error rate for Side=P is higher than for Side=S.

Age_group:
The highest error rate is observed in the "up to 15" age group —
about 22.8%.

Conclusion:
Tuned CatBoost continues to make an unequal number of errors
across different passenger groups.
The most noticeable differences are observed in HomePlanet, Deck,
and age groups.

When interpreting the error rate, the size of each group
must be taken into account.
'''


# Error analysis of Tuned CatBoost by categorical features

error_features = [
    "CryoSleep",
    "HomePlanet",
    "Deck",
    "VIP",
    "Side"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, feature in zip(axes, error_features):

    error_rate = (
        res_cat_tuned
        .groupby(feature)["error"]
        .mean()
        .sort_values(ascending=False)
    )

    error_rate.plot(
        kind="bar",
        ax=ax
    )

    ax.set_title(f"Error rate by {feature}")
    ax.set_xlabel(feature)
    ax.set_ylabel("Error rate")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)

axes[-1].axis("off")

plt.suptitle(
    "Error Analysis — Tuned CatBoost",
    fontsize=16
)

plt.tight_layout()
plt.show()


# Tuned CatBoost errors by age groups

age_error_cat = (
    res_cat_tuned
    .groupby("Age_group", observed=True)["error"]
    .mean()
)

plt.figure(figsize=(8, 5))

age_error_cat.plot(
    kind="bar"
)

plt.title("Tuned CatBoost Error Rate by Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Error rate")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


# Calculating permutation importance for Tuned CatBoost

perm_cat_tuned = permutation_importance(
    cat_model_tuned,
    X_val_cat,
    y_val,
    scoring="accuracy",
    n_repeats=10,
    random_state=42,
    n_jobs=1
)

importance_cat_tuned = pd.DataFrame({
    "feature": X_val_cat.columns,
    "importance_mean": perm_cat_tuned.importances_mean,
    "importance_std": perm_cat_tuned.importances_std
})

importance_cat_tuned = importance_cat_tuned.sort_values(
    "importance_mean",
    ascending=False
)

print("Permutation Importance — Tuned CatBoost:")
print(importance_cat_tuned)


'''
Permutation Importance estimates the change in model accuracy
when the values of a single feature are randomly shuffled.

The greater the decrease in accuracy after shuffling a feature,
the greater its contribution to the model's quality on the given dataset.

In the current experiment, the most noticeable contributions come from:
Spa, VRDeck, FoodCourt, CabinNumber, and RoomService.

VIP and CabinNumberMissing show the least influence.

It is important to note that permutation importance characterizes
a feature's contribution to a specific trained model
and does not indicate causality.
'''


# Permutation Importance — Tuned CatBoost

top_n = 15

plot_data = importance_cat_tuned.head(top_n).sort_values(
    "importance_mean"
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["feature"],
    plot_data["importance_mean"]
)

plt.xlabel("Mean decrease in accuracy")
plt.ylabel("Feature")
plt.title("Permutation Importance — Tuned CatBoost")

plt.tight_layout()
plt.show()


# Comparing permutation importance:
# Random Forest vs Tuned CatBoost

importance_compare = pd.merge(
    importance_df[["feature", "importance_mean"]].rename(
        columns={"importance_mean": "importance_rf"}
    ),
    importance_cat_tuned[["feature", "importance_mean"]].rename(
        columns={"importance_mean": "importance_catboost"}
    ),
    on="feature",
    how="outer"
).fillna(0)

importance_compare = importance_compare.sort_values(
    "importance_catboost",
    ascending=False
)

print("Feature importance comparison:")
print(importance_compare)


'''
The comparison of permutation importance shows
that Random Forest and Tuned CatBoost use features
with different relative importance.

For Tuned CatBoost, the most noticeable contributions come from
spending-related features: Spa, VRDeck, FoodCourt, and RoomService.

For Random Forest, the relative importance of CryoSleep is significantly higher.

This shows that different algorithms can use the same set of features
differently when making predictions.

However, differences in permutation importance do not mean
that one algorithm uses features better than the other.
'''


# Visual comparison of permutation importance
# Random Forest vs Tuned CatBoost

top_n = 15

plot_compare = importance_compare.head(top_n).copy()

plot_compare = plot_compare.sort_values(
    "importance_catboost"
)

y = np.arange(len(plot_compare))
height = 0.35

plt.figure(figsize=(11, 8))

plt.barh(
    y - height / 2,
    plot_compare["importance_rf"],
    height=height,
    label="Random Forest"
)

plt.barh(
    y + height / 2,
    plot_compare["importance_catboost"],
    height=height,
    label="CatBoost"
)

plt.yticks(
    y,
    plot_compare["feature"]
)

plt.xlabel("Permutation importance")
plt.ylabel("Feature")
plt.title("Random Forest vs CatBoost — Feature Importance")
plt.legend()

plt.tight_layout()
plt.show()


# Comparing baseline RF and tuned RF on cross-validation

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(rf_cv_scores) + 1),
    rf_cv_scores,
    marker="o",
    label="Baseline RF"
)

plt.plot(
    range(1, len(tuned_cv_scores) + 1),
    tuned_cv_scores,
    marker="o",
    label="Tuned RF"
)

plt.title("Baseline RF vs Tuned RF — Cross-Validation")
plt.xlabel("Fold")
plt.ylabel("Accuracy")

plt.xticks(
    range(1, len(rf_cv_scores) + 1)
)

plt.ylim(
    min(
        rf_cv_scores.min(),
        tuned_cv_scores.min()
    ) - 0.01,
    max(
        rf_cv_scores.max(),
        tuned_cv_scores.max()
    ) + 0.01
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Model comparison on Repeated Stratified K-Fold

cv_models = {
    "Random Forest": scores_e4_repeated,
    "Tuned Random Forest": scores_tuned_repeated,
    "CatBoost": scores_cat_repeated
}

cv_means = [
    np.mean(scores)
    for scores in cv_models.values()
]

cv_stds = [
    np.std(scores)
    for scores in cv_models.values()
]

'''
The plot shows the mean accuracy
for each model on the same splits of
Repeated Stratified K-Fold.

The vertical lines reflect the standard deviation
of results between splits.

The mean value characterizes the quality level,
while the standard deviation characterizes the variance of results.
'''

plt.figure(figsize=(10, 6))

bars = plt.bar(
    cv_models.keys(),
    cv_means,
    yerr=cv_stds,
    capsize=5
)

plt.title("Repeated Stratified K-Fold — Model Comparison")
plt.xlabel("Model")
plt.ylabel("Mean accuracy")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.3)

for bar, mean in zip(
    bars,
    cv_means
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{mean:.4f}",
        ha="center"
    )

plt.tight_layout()
plt.show()


# Distribution of Repeated Stratified K-Fold results

boxplot_data = [
    np.asarray(
        scores_e4_repeated,
        dtype=float
    ).ravel(),

    np.asarray(
        scores_tuned_repeated,
        dtype=float
    ).ravel(),

    np.asarray(
        scores_cat_repeated,
        dtype=float
    ).ravel()
]

'''
The boxplot shows the distribution of accuracy
across all splits of Repeated Stratified K-Fold.

The plot allows comparing the performance level
and its variance between splits.
'''

plt.figure(figsize=(10, 6))

plt.boxplot(
    boxplot_data,
    tick_labels=[
        "Random Forest",
        "Tuned Random Forest",
        "CatBoost"
    ]
)

plt.ylabel("Accuracy")
plt.title("Repeated Stratified K-Fold — Model Comparison")
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()


# Analysis of passenger spending distribution

plt.figure(figsize=(10, 6))

train[spend_cols].plot(
    kind="box",
    figsize=(10, 6)
)

plt.title("Passenger Spending Distribution")
plt.xlabel("Service Type")
plt.ylabel("Spending Amount")
plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


'''
The boxplot is used to compare the distribution of spending
across various ship services.

The plot allows evaluating median values,
data variance, and the presence of potential outliers.
'''


# Comparing mean spending between Transported classes

mean_spending = (
    train
    .groupby("Transported")[spend_cols]
    .mean()
    .T
)

plt.figure(figsize=(10, 6))

mean_spending.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Mean Spending by Transported Status")
plt.xlabel("Service Type")
plt.ylabel("Mean Spending")
plt.xticks(rotation=30)

plt.legend(
    ["False", "True"],
    title="Transported"
)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


'''
The plot compares the mean spending of passengers
for the two values of the target variable 'Transported'.

Differences in mean spending can be used
as additional information during feature analysis.

However, differences in mean values alone
do not indicate a causal relationship
between spending and the 'Transported' status.
'''


# Correlation matrix of numerical features

plt.figure(figsize=(12, 8))

corr = train[
    numeric_cols + ["Transported"]
].corr()

plt.imshow(
    corr,
    aspect="auto"
)

plt.colorbar()

plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(corr.columns)),
    corr.columns
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


'''
The correlation matrix shows the strength of the linear relationship
between numerical features and the target variable.

A positive correlation value indicates a direct relationship between features,
while a negative value indicates an inverse relationship.

Correlation does not imply causality
and does not reflect non-linear dependencies between features.
'''


# Comparing model accuracy on the validation set

model_names = [
    "Logistic Regression",
    "Random Forest",
    "Tuned Random Forest",
    "CatBoost",
    "Tuned CatBoost"
]

model_scores = [
    accuracy,
    accuracy_rf,
    accuracy_tuned,
    accuracy_cat,
    accuracy_cat_tuned
]

comparison_models = pd.DataFrame({
    "Model": model_names,
    "Accuracy": model_scores
})

print(comparison_models)


'''
We compare the accuracy of five models
on the same validation set.

The comparison includes:
- Logistic Regression
- Random Forest
- Tuned Random Forest
- CatBoost
- Tuned CatBoost

These results pertain to the current data split.
For a more reliable comparison, models are additionally
evaluated using cross-validation.
'''


# Accuracy comparison plot

plt.figure(figsize=(10, 6))

bars = plt.bar(
    comparison_models["Model"],
    comparison_models["Accuracy"]
)

plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.title("Model Quality Comparison")

plt.ylim(
    max(
        0,
        comparison_models["Accuracy"].min() - 0.03
    ),
    min(
        1,
        comparison_models["Accuracy"].max() + 0.02
    )
)

plt.xticks(rotation=20)


# Displaying exact accuracy values above the bars

for bar, score in zip(
    bars,
    comparison_models["Accuracy"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f"{score:.4f}",
        ha="center"
    )

plt.tight_layout()
plt.show()


'''
Final model comparison:

Results on the current validation set:

Logistic Regression      — 0.7883
Random Forest            — 0.8033
Tuned Random Forest      — 0.7956
CatBoost                 — 0.8087
Tuned CatBoost           — 0.8133

On the considered validation set, accuracy values differ between models.

For Random Forest, after hyperparameter tuning,
accuracy decreased from 0.8033 to 0.7956.

For CatBoost, after hyperparameter tuning,
accuracy increased from 0.8087 to 0.8133.

Thus, the impact of tuning on the current validation set
turned out to be different for the two models.

However, comparing based on only one validation set
is not sufficient to evaluate result stability.
Therefore, we additionally consider Repeated Stratified K-Fold.

CatBoost baseline:
mean = 0.8105
std  = 0.0059

Tuned CatBoost:
mean = 0.8100
std  = 0.0090

The mean accuracy values on repeated cross-validation
for the two models are practically identical.

At the same time, CatBoost baseline shows a slightly higher
mean accuracy and a lower standard deviation.
This means that on the considered splits,
the baseline results are more stable.

Conclusion:
On the current validation set, Tuned CatBoost shows
higher accuracy, but this advantage is not preserved
on Repeated Stratified K-Fold.

For further work, CatBoost baseline is used,
as it shows comparable quality with less result variance.

CatBoost baseline parameters:
iterations = 300
learning_rate = 0.1
depth = 6

Tuned CatBoost parameters are determined by the `best_params_cat` value
obtained during hyperparameter search.
'''

In [ ]:
# final_part
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool


def get_file_path(kaggle_path, local_path):
    if os.path.exists(kaggle_path):
        return kaggle_path
    return local_path


def main():
    train_path = get_file_path("/kaggle/input/competitions/spaceship-titanic/train.csv", "train.csv")
    test_path = get_file_path("/kaggle/input/competitions/spaceship-titanic/test.csv", "test.csv")

    print(f"Loading train data from: {train_path}")
    print(f"Loading test data from: {test_path}")

    test = pd.read_csv(test_path)
    train = pd.read_csv(train_path)

    spend_cols = [
        'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'
    ]

    categorical_cols = [
        'HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side'
    ]

    train['CabinNumber'] = train["Cabin"].str.split("/").str[1]
    train['Side'] = train["Cabin"].str.split("/").str[2]
    train["Deck"] = train["Cabin"].str.split("/").str[0]
    train["CabinNumber"] = pd.to_numeric(train["CabinNumber"])

    mask_true = train["CryoSleep"] == True
    mask_false = train["CryoSleep"] == False
    train.loc[mask_true, spend_cols] = train.loc[mask_true, spend_cols].fillna(0)
    for col in spend_cols:
        median = train.loc[mask_false, col].median()
        train.loc[mask_false, col] = train.loc[mask_false, col].fillna(median)

    cryo_na = train["CryoSleep"].isna()
    total_spend = train[spend_cols].fillna(0).sum(axis=1)

    mask_false = cryo_na & (total_spend > 0)
    train.loc[mask_false, "CryoSleep"] = False

    mask_true = (
        train["CryoSleep"].isna()
        & train[spend_cols].notna().all(axis=1)
        & (train[spend_cols].sum(axis=1) == 0)
    )
    train.loc[mask_true, "CryoSleep"] = True

    mask_11 = (
        train["CryoSleep"].isna()
        & (train[spend_cols].fillna(0).sum(axis=1) == 0)
    )
    train.loc[mask_11, spend_cols] = train.loc[mask_11, spend_cols].fillna(0)

    mask_false = train["CryoSleep"] == False
    for col in spend_cols:
        median = train.loc[mask_false, col].median()
        train.loc[mask_false, col] = train.loc[mask_false, col].fillna(median)

    train["CryoSleep"] = train["CryoSleep"].fillna(train["CryoSleep"].mode()[0]).astype(bool)
    train["Age"] = train["Age"].fillna(train["Age"].median())
    train["VIP"] = train["VIP"].fillna(False).astype(bool)

    train["HasSpend"] = (train[spend_cols].sum(axis=1) > 0).astype(int)

    deck_to_planet = {
        "A": "Europa", "B": "Europa", "C": "Europa",
        "G": "Earth", "T": "Europa"
    }
    mask_planet = train["HomePlanet"].isna()
    deck = train["Cabin"].str.split("/").str[0]
    planet = deck.map(deck_to_planet).fillna("Unknown")
    train.loc[mask_planet, "HomePlanet"] = planet[mask_planet]

    train["Destination"] = train["Destination"].fillna("Unknown")
    train["CabinNumberMissing"] = train["CabinNumber"].isna().astype(int)
    train["CabinNumber"] = train["CabinNumber"].fillna(train["CabinNumber"].median())
    train["Deck"] = train["Deck"].fillna("Unknown")
    train["Side"] = train["Side"].fillna("Unknown")

    cols_to_drop = ["Cabin", "Name", "PassengerId"]
    train = train.drop(columns=cols_to_drop, errors="ignore")

    test_final = test.copy()
    test_passenger_id = test_final["PassengerId"].copy()

    test_final["Deck"] = test_final["Cabin"].str.split("/").str[0]
    test_final["CabinNumber"] = test_final["Cabin"].str.split("/").str[1]
    test_final["CabinNumber"] = pd.to_numeric(test_final["CabinNumber"], errors="coerce")
    test_final["Side"] = test_final["Cabin"].str.split("/").str[2]

    cryo_na_test = test_final["CryoSleep"].isna()
    total_spend_test = test_final[spend_cols].fillna(0).sum(axis=1)

    mask_false_test = cryo_na_test & (total_spend_test > 0)
    test_final.loc[mask_false_test, "CryoSleep"] = False

    mask_true_test = (
        test_final["CryoSleep"].isna()
        & test_final[spend_cols].notna().all(axis=1)
        & (test_final[spend_cols].sum(axis=1) == 0)
    )
    test_final.loc[mask_true_test, "CryoSleep"] = True

    test_final["CryoSleep"] = test_final["CryoSleep"].fillna(train["CryoSleep"].mode()[0]).astype(bool)

    mask_true_test = test_final["CryoSleep"] == True
    test_final.loc[mask_true_test, spend_cols] = test_final.loc[mask_true_test, spend_cols].fillna(0)

    mask_false_test = test_final["CryoSleep"] == False
    for col in spend_cols:
        median = train.loc[train["CryoSleep"] == False, col].median()
        test_final.loc[mask_false_test, col] = test_final.loc[mask_false_test, col].fillna(median)

    test_final["Age"] = test_final["Age"].fillna(train["Age"].median())
    test_final["VIP"] = test_final["VIP"].fillna(False).astype(bool)
    test_final["HasSpend"] = (test_final[spend_cols].sum(axis=1) > 0).astype(int)

    mask_planet_test = test_final["HomePlanet"].isna()
    planet_test = test_final["Deck"].map(deck_to_planet).fillna("Unknown")
    test_final.loc[mask_planet_test, "HomePlanet"] = planet_test[mask_planet_test]

    test_final["Destination"] = test_final["Destination"].fillna("Unknown")
    test_final["CabinNumberMissing"] = test_final["CabinNumber"].isna().astype(int)
    test_final["CabinNumber"] = test_final["CabinNumber"].fillna(train["CabinNumber"].median())
    test_final["Deck"] = test_final["Deck"].fillna("Unknown")
    test_final["Side"] = test_final["Side"].fillna("Unknown")

    cols_to_drop = ["Cabin", "Name", "PassengerId"]
    test_final = test_final.drop(columns=cols_to_drop, errors="ignore")

    print("\nMissing values in test:")
    print(test_final.isna().sum())
    print("\nTest set shape:")
    print(test_final.shape)

    X_full_cat = train.drop(columns="Transported").copy()
    y_full = train["Transported"].copy()
    X_test_cat = test_final.copy()

    for col in categorical_cols:
        X_full_cat[col] = X_full_cat[col].astype(str)
        X_test_cat[col] = X_test_cat[col].astype(str)

    X_test_cat = X_test_cat[X_full_cat.columns]

    train_pool = Pool(
        data=X_full_cat,
        label=y_full,
        cat_features=categorical_cols
    )

    test_pool = Pool(
        data=X_test_cat,
        cat_features=categorical_cols
    )

    final_cat_model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    final_cat_model.fit(train_pool)

    test_predictions = final_cat_model.predict(test_pool)
    test_predictions = np.asarray(test_predictions).ravel().astype(bool)

    submission = pd.DataFrame({
        "PassengerId": test_passenger_id,
        "Transported": test_predictions
    })

    print("\nFirst rows of submission:")
    print(submission.head())
    print("\nSubmission shape:")
    print(submission.shape)
    print("\nNumber of predictions:")
    print(submission["Transported"].value_counts())

    submission_dir = "/kaggle/working"
    if not os.path.exists(submission_dir):
        submission_dir = "."

    submission_path = os.path.join(submission_dir, "submission.csv")
    submission.to_csv(submission_path, index=False)

    print(f"\n✅ File successfully saved to: {os.path.abspath(submission_path)}")


if __name__ == "__main__":
    main()